<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff; 
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); 
        font-weight: bold; 
        margin-bottom: 10px; 
        font-size: 36px; 
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        🛒 Brazilian E-Commerce Analysis 🔍
    </h1>
</div>

# 📂 Overview

The **Brazilian E-Commerce Public Dataset by Olist** is a real-world dataset from the Olist marketplace in Brazil. It records over **100,000 orders** placed between **2016 and 2018**, including detailed information about **customers, sellers, products, payments, deliveries, and reviews**.

The dataset consists of multiple interconnected CSV files — such as `orders`, `customers`, `sellers`, `order_items`, `payments`, `products`, and `reviews` — enabling end-to-end analysis of the order journey: from purchase and payment to delivery and customer feedback.

Typical use cases include analyzing customer behavior, seller performance, delivery times, payment trends, and building predictive models such as review score prediction or late delivery forecasting.


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #FFFFFF; 
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); 
        font-weight: bold; 
        margin-bottom: 5px; 
        font-size: 28px; 
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [2]:
# Core data manipulation libraries
import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 500) # To display all the columns of dataframe
pd.set_option("max_colwidth", None) # To set the width of the column to maximum

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #FFFFFF; 
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); 
        font-weight: bold; 
        margin-bottom: 5px; 
        font-size: 28px; 
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Load Data
    </h1>
</div>


In [3]:
# Load the datasets
df_customers = pd.read_csv("olist_customers_dataset.csv")
df_geolocation = pd.read_csv("olist_geolocation_dataset.csv")
df_order_items = pd.read_csv("olist_order_items_dataset.csv")
df_order_payments = pd.read_csv("olist_order_payments_dataset.csv")
df_order_reviews = pd.read_csv("olist_order_reviews_dataset.csv")
df_orders = pd.read_csv("olist_orders_dataset.csv")
df_products = pd.read_csv("olist_products_dataset.csv")
df_sellers = pd.read_csv("olist_sellers_dataset.csv")
df_product_category_name_translation = pd.read_csv("product_category_name_translation.csv")

# Verify shapes
print("Customer Data Shape:", df_customers.shape)
print("\nGeolocation Data Shape:", df_geolocation.shape)
print("\nOrder Items Data Shape:", df_order_items.shape)
print("\nOrder Payment Data Shape:", df_order_payments.shape)
print("\nOrder Review Data Shape:", df_order_reviews.shape)
print("\nOrders Data Shape:", df_orders.shape)
print("\nProducts Data Shape:", df_products.shape)
print("\nSellers Data Shape:", df_sellers.shape)
print("\nProduct Category Name Data Shape:", df_product_category_name_translation.shape)

Customer Data Shape: (99441, 5)

Geolocation Data Shape: (1000163, 5)

Order Items Data Shape: (112650, 7)

Order Payment Data Shape: (103886, 5)

Order Review Data Shape: (99224, 7)

Orders Data Shape: (99441, 8)

Products Data Shape: (32951, 9)

Sellers Data Shape: (3095, 4)

Product Category Name Data Shape: (71, 2)


In [ ]:
timestamp_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", 
                  "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in timestamp_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [10]:
# Create column total_value
df_order_items["total_value"] = df_order_items["price"] + df_order_items["freight_value"]

df_order_items_ = df_order_items.groupby(by = ['order_id', 'product_id', 'shipping_limit_date'], group_keys = False)[['order_id', 'product_id', 'price', 'total_value']]\
    .agg(
        product_counts = pd.NamedAgg (column = "product_id", aggfunc = "count"),
        total_price = pd.NamedAgg (column = "price", aggfunc = "sum"),
        total_value = pd.NamedAgg (column = "total_value", aggfunc = "sum")
    )\
    .reset_index()




In [13]:
df_order_items_.head()

,order_id,product_id,shipping_limit_date,product_counts,total_price,total_value
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,2017-09-19 09:45:35,1,58.90,72.19
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,2017-05-03 11:05:13,1,239.90,259.83
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,2018-01-18 14:48:30,1,199.00,216.87
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,2018-08-15 10:10:18,1,12.99,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,2017-02-13 13:57:51,1,199.90,218.04
